# Wind correction

The metamodel cases (notebook 05) run **without wind**. The reconstruction
(`mm.reconstruct(..., wind_correction=True)`) adds the local wind sea once per
time step and site, before the goals' spectra are summed:

- contributions whose *offshore* forcing is a wind sea (steep, broad; weight
  `w` from `metamodel.wind_correction.steepness_range` / `spr_range`) give the
  wind-sea energy at the site, `E_ws = Σ w·hs²`;
- `K = Hs(with wind) / Hs(without wind)` comes from a model fitted here, on
  dimensionless wind-sea Hs, fetch and depth (`g·X/U²`) and the wind-wave angle;
- one extra contribution, `(K² − 1)·E_ws` from the wind direction, joins the sum.

Workflow:

1. **Fetch table**: fetch and mean depth along it per output site and wind
   direction, over the model grid (`outputs/wind/fetch_table_{model}.nc`).
2. **Training cases**: `num_cases` full-domain wind seas (every goal forced the
   same), each with a wind of `speed_ratio` × the fully developed speed blowing
   `wind_wave_angle` off the waves; built twice, with and without wind, with
   the model's dynamic templates (`simulations.<model>.wind_cases`).
3. Run them (on the cluster), postprocess.
4. **Fit** `K`, score it on held-out cases, save
   (`outputs/wind/wind_correction_{model}.pkl`).

Set `metamodel.wind_correction.enabled = true` to use it by default in 09–12.

In [ ]:
import matplotlib.pyplot as plt
from utils import simulation as sim
from utils import wind
from utils.config import load_goals, simulation_model

In [ ]:
model = simulation_model()
cfg = wind.config()
print(f"model      : {model}")
print(f"cases      : {wind.wind_cases_dir(True).parent}")
print(f"wind       : {cfg['wind']} (reconstruction), added Tp: {cfg['added_tp']}")
print(
    f"design     : {cfg['num_cases']} cases, U/U_fd {cfg['speed_ratio']}, angle {cfg['wind_wave_angle']}"
)
print(f"wind sea   : steepness {cfg['steepness_range']}, spr {cfg['spr_range']}")

## 1 · Fetch table

In [ ]:
table = wind.fetch_table()  # cached; recompute=True after changing sites or grid
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, direction in zip(axes, (270.0, 330.0)):
    sc = ax.scatter(
        table["point_x"],
        table["point_y"],
        c=table["fetch_eff"].sel(dir=direction) / 1e3,
        s=12,
        cmap="viridis",
    )
    plt.colorbar(sc, ax=ax, label="effective fetch (km)")
    ax.set_title(f"Wind from {direction:.0f}°")
    ax.set_aspect("equal")
plt.show()
print(f"sites outside the grid (no correction): {int(table.site_depth.isnull().sum())}")

## 2 · Training cases (with and without wind)

In [ ]:
goals = load_goals()
cases = wind.generate_wind_cases(goals)
cases.describe().round(2)

In [ ]:
setup = sim.HyWavesWrapperSetup.from_case(goals)
wrappers = wind.wind_case_wrappers(setup, cases)  # also writes wind_cases.csv
for wrapper in wrappers.values():
    wrapper.build_cases()
# after a kernel restart: wrappers = wind.wind_case_wrappers(setup), then
# wrapper.load_cases() for each

## 3 · Run (by hand, on the cluster), postprocess

SnapWave with wind needs the `wind1` build (`simulations.snapwave.slurm.setup`),
`sector = 360` and `niter = 200` (set by the dynamic template when `u10 > 0`):
minutes per case instead of seconds. SWAN with wind (`GEN3`) is slower too.

In [ ]:
# wrappers["wind"].run_cases_slurm(
#     **sim.slurm_kwargs(cpus_per_task=12, mem="16gb", extra_sbatch=["--time=06:00:00"])
# )
# wrappers["no_wind"].run_cases_slurm(**sim.slurm_kwargs())

In [ ]:
for label, wrapper in wrappers.items():
    display(label, wrapper.monitor_cases(value_counts="simple"))

In [ ]:
for label, wrapper in wrappers.items():
    wrapper.postprocess_cases().to_netcdf(wind.postprocessed_path(label == "wind"))

## 4 · Fit K

Scores: Hs on held-out training cases (reference = the run **with** wind),
without and with the correction.

In [ ]:
correction, scores = wind.fit_wind_correction()
scores.pivot_table(index="group", columns="estimate", values=["bias", "rmse"]).round(3)

In [ ]:
pairs = wind.training_pairs().dropna(subset=["hs0", "hs1"])
pairs = pairs[pairs["hs0"] > 0.1]
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    9.81 * pairs["hs0"] / pairs["u10"] ** 2,
    pairs["hs1"] / pairs["hs0"],
    c=pairs["depth"],
    s=4,
    cmap="viridis_r",
    vmax=40,
)
plt.colorbar(sc, label="site depth (m)")
ax.axhline(1, color="k", lw=0.8)
ax.set(
    xlabel="g Hs / U² (without wind)", ylabel="K = Hs with / without wind", xscale="log"
)
plt.show()

## 5 · How K is estimated: growth formula vs fitted model

`K` is the factor the wind sea grows by when the wind blows (Hs with wind /
Hs without). There are two ways to estimate it:

- **Growth formula** (textbook, Kahma & Calkoen): treat the wind sea that
  arrives without wind (`Hs_ws`) as a sea already grown over some distance
  (its *equivalent fetch*), let it grow over the fetch left to the site, and
  read the new height; `K` = new / old. Optionally cap it with the shallow-water
  limit at the site (Breugem & Holthuijsen). These are the curves SnapWave's
  own wind source uses.
- **Fitted model** (what the reconstruction uses): learned from these training
  runs, with the same physical inputs made dimensionless with the wind speed
  `U`: `g·Hs_ws/U²` (how developed the wind sea is; 0.24 = fully developed),
  `g·F/U²` (fetch left), `g·d/U²` and `g·d̄/U²` (depth at the site and along
  the fetch) and the wind–wave angle.

Below, on training cases held out from the fit, both are compared with the `K`
the model runs actually show (reference, x-axis). The formula has no way to
know that over shallow seas the wind also keeps the existing storm sea from
decaying, and it overshoots for very young seas; the fitted model learns
both from the runs.

In [ ]:
from bluemath_tk.waves.hywaves.plots import plot_wind_k_comparison

k_df = (
    wind.k_comparison()
)  # held-out cases of fit_wind_correction(); the model is refitted without them
fig = plot_wind_k_comparison(k_df)
plt.show()
k_df[["k_runs", "k_fitted", "k_formula_deep", "k_formula_depth"]].describe().round(3)

## 6 · Runs side by side: what the wind adds and what the correction adds

Held-out training cases (the model never saw them, as in §5), at every output
site:

- what the wind adds in the runs (with − without wind) and what the correction
  adds (corrected − without wind), on the same scale;
- their difference, corrected − with wind: the error of the correction;
- every site: the run with wind (x, reference) against the run without wind and
  the corrected one.

`holdout_cases()` lists the cases to pick from; `hs_ws_u2` = g·Hs_ws/U² offshore
(how developed the wind sea is, 0.24 = fully developed).

Keep in mind: the correction only adds energy (`K < 1` adds nothing), while the
runs with wind can be lower than without at some sites; and the runs with wind
use other SnapWave settings (`sector = 360`, `niter = 200`), so part of that
difference may be numerical.

In [ ]:
held = wind.holdout_cases()
held.round(2).sort_values("hs_ws_u2")

In [ ]:
from bluemath_tk.waves.hywaves.plots import plot_wind_case_comparison

case_ids = list(held.sort_values("hs_ws_u2").index[[0, len(held) // 2, -1]])  # young -> developed
fig = plot_wind_case_comparison(wind.case_comparison(case_ids))
plt.show()

## 7 · How the model corrects: K against each input

`K` of the fitted model at a few sites for one wind direction, varying one input
at a time around a base state (`wind_dir`, `hs_ws`, `u10`, `angle`): wind speed,
wind–wave angle, and how developed the wind sea is (g·Hs_ws/U²; 0.24 = fully
developed). Shaded: outside the training cases (1–99 % at the sites), where the
model keeps its edge value. This is where real hours that look nothing like the
training cases (decaying storms, weak winds over a steep sea) get their `K`.

In [ ]:
from bluemath_tk.waves.hywaves.plots import plot_k_response

depth = wind.fetch_table()["site_depth"].to_series().dropna()
sites = [depth.sub(d).abs().idxmin() for d in (5.0, 10.0, 20.0, 30.0)]  # sites closest to these depths
curves = wind.k_response(sites, wind_dir=300.0, hs_ws=2.0, u10=15.0, angle=0.0)
fig = plot_k_response(curves, wind.training_range())
plt.show()